In [ ]:
from DeepLearning.Environments.CatanEnv import CatanTradingEnv
from DeepLearning.CustomMaskablePPO import MaskablePPO
from Client.DeepLearning.Encoders.MainGame.ActionMask.GetActionMask import getActionMaskTrading
from DeepLearning.GetObservation import getObservationTrading
import os

os.environ["UPDATE_MODELS_UNIFORM"] = "False"
os.environ["UPDATE_MODELS_DIST"] = "False"
os.environ["MODEL_NAME"] = "None"
os.environ["MODEL_1_NAME"] = ""
os.environ["MODEL_2_NAME"] = ""
os.environ["MODEL_3_NAME"] = ""

env = CatanTradingEnv(trading=True)
actionMask = getActionMaskTrading
observation = getObservationTrading
gamma = 0.99

info = {
    "env": "CatanEnv",
    "Timesteps": "1M",
    "Opponents": "self.play",
    "Rewards": "Setup, Dense, Bank Trades"
}
name = "Trading_20Turns_CitySettlement_SmallTrading"

netArchDict = dict(pi=[128, 128], vf=[128, 128])

# model = MaskablePPO("MlpPolicy", env, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, verbose=1, getActionMask=actionMask, getObservation=observation, info=info, saveName=name, tensorboard_log="./tensorboard_logs/")
model = MaskablePPO.load("DeepLearning/Models/Trading_20Turns_CitySettlement/Trading_20Turns_CitySettlement_7M.zip", env=env)
model.saveName = name
model.learn(total_timesteps=32_000_000, tb_log_name=f"{name}")
# model.save("DeepLearning/Models/TradingBase_PlayerTradingAdded_20Turns/Final")

In [ ]:
"""
Running Agent simulations
"""
from Agents.AgentRandom2 import AgentRandom2
from Agents.AgentModel import AgentModel
from DeepLearning.CustomMaskablePPO import MaskablePPO
from Game.CatanPlayer import PlayerStatsTracker

# best_model = AgentMultiModel("P1", 1, model=MaskablePPO.load("DeepLearning/Models/NoSetup/NoSetupDenseRewardEnv-10M.zip"), setupModel=MaskablePPO.load("DeepLearning/Models/Setup/SetupRandom_wins_1M.zip"), fullSetup=False)

winner = [0,0,0,0]
player0Stats = PlayerStatsTracker()
Player0LosingStats = PlayerStatsTracker()
player1Stats = PlayerStatsTracker()

players = [ AgentModel("P0", 0, recordStats=True, playerTrading=True, model=MaskablePPO.load("DeepLearning/Models/Trading_20Turns_CitySettlement/Trading_20Turns_CitySettlement_6899712.zip")),
            AgentRandom2("P1", 1, recordStats=True, playerTrading=True),
            AgentRandom2("P2", 2, recordStats=True, playerTrading=True),
            AgentRandom2("P3", 3, recordStats=True, playerTrading=True),]
'''
COLLECT_STATS = True
for episode in range(10):
    game = CreateGame(players)
    game = pickle.loads(pickle.dumps(game, -1))
    numTurns = 0
    while True:
        currPlayer = game.gameState.players[game.gameState.currPlayer]

        agentAction = currPlayer.DoMove(game)
        agentAction.ApplyAction(game.gameState)

        if currPlayer.seatNumber == 0 and agentAction.type == 'EndTurn':
        #     DisplayImage(game.gameState, agentAction)
        #     time.sleep(1)
            numTurns += 1

        if game.gameState.currState == "OVER" or numTurns >= 20:
            DisplayImage(game.gameState, agentAction)
            break
    
    # print("Winner: ", game.gameState.winner)
    winner[game.gameState.winner] += 1
    lost = game.gameState.winner != 0

    # Stats
    if COLLECT_STATS:
        game.gameState.players[0].generatePlayerStats()
        game.gameState.players[1].generatePlayerStats()

        player0Stats += game.gameState.players[0].stats
        player1Stats += game.gameState.players[1].stats
        if lost:
            Player0LosingStats += game.gameState.players[0].stats

# Collect stats
if COLLECT_STATS:
    player0Stats.getAverages()
    Player0LosingStats.getAverages()
    player1Stats.getAverages()
    player0Data = player0Stats.getList()
    player0LosingData = Player0LosingStats.getList()
    player1Data = player1Stats.getList()

    p_hat0 = winner[0] / sum(winner)
    p_hat1 = winner[1] / sum(winner)
    margin_error0 = round(100*(1.96 * math.sqrt((p_hat0 * (1 - p_hat0)) / sum(winner))), 2)
    margin_error1 = round(100*(1.96 * math.sqrt((p_hat1 * (1 - p_hat1)) / sum(winner))), 2)
    player0Data.insert(0, margin_error0)
    player0LosingData.insert(0, -1)
    player1Data.insert(0, margin_error1)
    player0Data.insert(0, winner[0]/sum(winner))
    player0LosingData.insert(0, -1)
    player1Data.insert(0, winner[1]/sum(winner))
    player0Data.insert(0, "Player0")
    player0LosingData.insert(0, "Player0LossesStats")
    player1Data.insert(0, "Player1")

    table = tabulate([player0Data, player0LosingData, player1Data], headers=headers, tablefmt='simple')
    print(table)

print(f"\nNum turns: {numTurns}")

print("\n\nWinnings: ", winner)


# Brick, ore, wool, wheat, wood
'''

In [ ]:
# # Save to csv
# fileName = f'GammaTest_winReward_99_v_3Random.csv'
# df = pd.DataFrame([player0Data, player0LosingData, player1Data], columns=headers)
# df.to_csv(f'DeepLearning/Data/Hyperparameters/{fileName}', index=False)

# from DeepLearning.GetActionMask import allActionsDict

# print(allActionsDict)

# model.save("DeepLearning/Models/Trading_20Turns_CitySettlement_SmallTrading/Final")
